In [ ]:
!git clone https://github.com/chaturvediabhay24/vSLM-From-Scratch.git

In [ ]:
PATH_TO_REPO = "/content/vSLM-From-Scratch/"

import sys
sys.path.insert(0, PATH_TO_REPO)

import glob
import numpy as np
import pandas as pd
from pathlib import Path

from src.preprocessing.preprocessing import PreProcessing
from src.tokenization.tokenizer import BPETokenizer

### Download TinyStories dataset

Downloads from HuggingFace (`roneneldan/TinyStories`) and saves as `train.csv` with a `text` column.  
Skips if the file already exists.

In [ ]:
import os

csv_path = PATH_TO_REPO + "dataset/tiny-stories/train.csv"

if not os.path.exists(csv_path):
    !pip install -q datasets
    from datasets import load_dataset

    print("Downloading TinyStories from HuggingFace...")
    ds = load_dataset("roneneldan/TinyStories", split="train")

    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    ds.to_csv(csv_path, index=False)
    print(f"Saved {len(ds):,} stories to {csv_path}")
else:
    print(f"train.csv already exists at {csv_path} — skipping download")

### Google Drive Persistence

Mount Google Drive to persist tokenizer and encoded token parts across Colab sessions.  
The training notebook loads from the same `GDRIVE_DATA_DIR`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

GDRIVE_DATA_DIR = Path("/content/drive/MyDrive/vSLM-data")
GDRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data dir: {GDRIVE_DATA_DIR}")

In [ ]:
import csv

# --- Tiny Stories (sampled fraction for tokenizer training) ---
preprocessor = PreProcessing(PATH_TO_REPO + "dataset/tiny-stories/train.csv")
preprocessor.load_data().sample_corpus(n=100_000, random_state=42)

print(f"Tiny Stories: {len(preprocessor.data):,} stories (sampled 100K for tokenizer)")
print(f"Tiny Stories corpus: {len(preprocessor.corpus):,} characters")

# --- Personal Details (complete — all Q&A CSVs) ---
# Some CSVs have unquoted commas in the answer field, so we read with
# csv.reader and rejoin anything after the first column.
personal_dir = PATH_TO_REPO + "dataset/personal-details"
personal_dfs = []
for csv_file in sorted(glob.glob(f"{personal_dir}/*.csv")):
    with open(csv_file, "r") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = []
        for row in reader:
            if len(row) >= 2:
                rows.append({"question": row[0], "answer": ",".join(row[1:])})
    df = pd.DataFrame(rows)
    personal_dfs.append(df)
    print(f"  {Path(csv_file).name}: {len(df)} rows")

personal_df = pd.concat(personal_dfs, ignore_index=True)
print(f"\nTotal personal-details Q&A pairs: {len(personal_df)}")

# Format Q&A pairs as training text
personal_texts = []
for _, row in personal_df.iterrows():
    q = str(row["question"]).strip()
    a = str(row["answer"]).strip()
    personal_texts.append(f"Question: {q} Answer: {a}")

personal_corpus = "<EOS>".join(personal_texts) + "<EOS>"
print(f"Personal corpus: {len(personal_corpus):,} characters ({len(personal_corpus) / 1e3:.1f} KB)")

# --- Combined corpus for tokenizer training ---
combined_corpus = personal_corpus + preprocessor.corpus

print(f"\nCombined tokenizer training corpus: {len(combined_corpus):,} characters")
print(f"  Personal: {len(personal_corpus):,} ({100*len(personal_corpus)/len(combined_corpus):.2f}%)")
print(f"  Stories:  {len(preprocessor.corpus):,} ({100*len(preprocessor.corpus)/len(combined_corpus):.2f}%)")

In [ ]:
tokenizer = BPETokenizer(vocab_size=4096, special_tokens=["<EOS>"])
tokenizer.train(combined_corpus, verbose=True)

In [5]:
print(f"Vocabulary size: {len(tokenizer.vocab)}")
print(f"Number of merges: {len(tokenizer.merges)}")

def display_token(t):
    return t.decode("utf-8", errors="replace") if isinstance(t, bytes) else t

print("\nFirst 20 merges (most common byte pairs):")
for i, merge in enumerate(tokenizer.merges[:20]):
    merged = tokenizer.vocab[257 + i]
    print(f"  {i}: '{display_token(tokenizer.vocab[merge[0]])}' + "
          f"'{display_token(tokenizer.vocab[merge[1]])}' -> '{display_token(merged)}'")

print(f"\nLast 20 merges (longest learned tokens):")
for i in range(max(0, len(tokenizer.merges) - 20), len(tokenizer.merges)):
    merge = tokenizer.merges[i]
    merged = tokenizer.vocab[257 + i]
    print(f"  {i}: '{display_token(tokenizer.vocab[merge[0]])}' + "
          f"'{display_token(tokenizer.vocab[merge[1]])}' -> '{display_token(merged)}'")

Vocabulary size: 4096
Number of merges: 3839

First 20 merges (most common byte pairs):
  0: 'h' + 'e' -> 'he'
  1: ' ' + 't' -> ' t'
  2: ' ' + 'a' -> ' a'
  3: ' ' + 's' -> ' s'
  4: ' ' + 'w' -> ' w'
  5: 'n' + 'd' -> 'nd'
  6: ' t' + 'he' -> ' the'
  7: 'e' + 'd' -> 'ed'
  8: ' a' + 'nd' -> ' and'
  9: ' t' + 'o' -> ' to'
  10: ' ' + 'b' -> ' b'
  11: 'i' + 'n' -> 'in'
  12: ' ' + 'h' -> ' h'
  13: ' w' + 'a' -> ' wa'
  14: 'r' + 'e' -> 're'
  15: 'o' + 'u' -> 'ou'
  16: ' ' + 'f' -> ' f'
  17: 'i' + 't' -> 'it'
  18: ' ' + 'c' -> ' c'
  19: ' ' + 'l' -> ' l'

Last 20 merges (longest learned tokens):
  3819: ' climb' + 'ing' -> ' climbing'
  3820: ' tr' + 'ump' -> ' trump'
  3821: ' me' + 'at' -> ' meat'
  3822: ' b' + 'ur' -> ' bur'
  3823: ' Joe' + 'y' -> ' Joey'
  3824: ' c' + 'elery' -> ' celery'
  3825: ' spl' + 'it' -> ' split'
  3826: ' c' + 'ord' -> ' cord'
  3827: ' p' + 'ed' -> ' ped'
  3828: ' listen' + 'ing' -> ' listening'
  3829: ' ph' + 'ot' -> ' phot'
  3830: ' h' +

In [ ]:
test_texts = [
    "Once upon a time, there was a little girl.",
    "The cat sat on the mat.",
    "Hello<EOS>World",
    'She said, "I love you!"',
    "Question: Who is Abhay Chaturvedi? Answer: Abhay is an AI/ML Engineer.",
]

for text in test_texts:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    print(f"Original:   {text!r}")
    print(f"Token IDs:  {ids}")
    print(f"Decoded:    {decoded!r}")
    print(f"Num tokens: {len(ids)}")
    print(f"Roundtrip:  {'OK' if text == decoded else 'MISMATCH'}")
    print()

In [ ]:
# Compression ratio on held-out stories
eval_preprocessor = PreProcessing(PATH_TO_REPO + "dataset/tiny-stories/train.csv")
eval_preprocessor.load_data().sample_corpus(n=1_000, random_state=99)

encoded = tokenizer.encode(eval_preprocessor.corpus)
num_bytes = len(eval_preprocessor.corpus.encode("utf-8"))
num_tokens = len(encoded)
ratio = num_bytes / num_tokens

print(f"Eval corpus: {num_bytes:,} bytes")
print(f"Encoded: {num_tokens:,} tokens")
print(f"Compression ratio: {ratio:.2f} bytes/token")

In [ ]:
tokenizer.save(str(GDRIVE_DATA_DIR / "tokenizer.json"))
print(f"Tokenizer saved to {GDRIVE_DATA_DIR / 'tokenizer.json'}")

In [ ]:
# loaded = BPETokenizer.load(str(GDRIVE_DATA_DIR / "tokenizer.json"))
loaded = BPETokenizer.load("dataset/tokenizer.json")

test = "Once upon a time<EOS>"
assert loaded.encode(test) == tokenizer.encode(test)
assert loaded.decode(loaded.encode(test)) == test
print("Load/save roundtrip verified!")

In [ ]:
from src.tokenization.corpus_encoder import CorpusEncoder

tokenizer = BPETokenizer.load(str(GDRIVE_DATA_DIR / "tokenizer.json"))
encoder = CorpusEncoder(tokenizer, parts_dir=str(GDRIVE_DATA_DIR / "tokens_parts"))

# Encode in parallel — resumes from already-saved parts
# 8 workers on a 10-core machine (leaves 2 cores for system/Jupyter)
encoder.encode_parts(
    csv_path=PATH_TO_REPO + "dataset/tiny-stories/train.csv",
    stories_per_chunk=5_000,
    n_workers=8,
)

### Encode Personal Details (with repetition for training emphasis)

Personal details are a tiny dataset (~300 Q&A pairs). We repeat them **N times** so they
represent a meaningful fraction of training data. Parts are saved to a separate directory
(`tokens_parts_personal/`) to keep them distinct from Tiny Stories parts.

Adjust `PERSONAL_REPEATS` to control how heavily the model is trained on personal details.

In [ ]:
PERSONAL_REPEATS = 50  # repeat so model sees personal details frequently

personal_corpus_full = "<EOS>".join(personal_texts) + "<EOS>"
personal_ids = tokenizer.encode(personal_corpus_full)
personal_arr = np.array(personal_ids, dtype=np.uint16)

# Tile and save as parts in a separate directory
personal_repeated = np.tile(personal_arr, PERSONAL_REPEATS)
personal_parts_dir = GDRIVE_DATA_DIR / "tokens_parts_personal"
personal_parts_dir.mkdir(parents=True, exist_ok=True)

# Save as ~1M token parts (matching Tiny Stories part size)
tokens_per_part = 1_000_000
n_parts = max(1, (len(personal_repeated) + tokens_per_part - 1) // tokens_per_part)

for i in range(n_parts):
    start = i * tokens_per_part
    end = min(start + tokens_per_part, len(personal_repeated))
    np.save(personal_parts_dir / f"part_{i:05d}.npy", personal_repeated[start:end])

print(f"Personal details: {len(personal_arr):,} tokens × {PERSONAL_REPEATS} = {len(personal_repeated):,} tokens")
print(f"Saved as {n_parts} part(s) to {personal_parts_dir}")

In [ ]:
# Run this after all parts are encoded
tokens = encoder.merge_parts(str(GDRIVE_DATA_DIR / "tokens.npy"))

In [ ]:
print("Done")

In [ ]:
0